## Set Up Environment

#### Imports

In [35]:
# import libraries
from __future__ import annotations
import matplotlib.pyplot as plt
from typing import Iterable
from pathlib import Path
import seaborn as sns
import pandas as pd
import polars as pl
import numpy as np
import random
import os

#### Configuration for File Reading

In [36]:
# File paths for reading data
NOTEBOOK_DIRECTORY = Path.cwd()
RAW_DATA_DIRECTORY = NOTEBOOK_DIRECTORY / "../Data/Raw"
DATASET_FILENAME = "diabetic_data.csv"
MISSING_VALUE_TOKENS = "?"
DATASET_PATH = (RAW_DATA_DIRECTORY / DATASET_FILENAME).resolve()

#### Load Raw Data

In [37]:
def load_diabetes_dataset(csv_path: Path) -> pd.DataFrame:
    """
    Load the diabetes readmission dataset
    """
    dataset = pd.read_csv(
        csv_path,
        na_values=MISSING_VALUE_TOKENS,
        low_memory=False,
    )
    return dataset

# raw use
df_raw = load_diabetes_dataset(DATASET_PATH)

## Targets, Missing Values and Duplicates

#### Configuration for Cleaning

In [38]:
# target encoding
READMISSION_TARGET_COLUMN = "readmitted"
BINARY_TARGET_COLUMN = "readmitted_30_days"
TARGET_MAPPING = {
    "<30": 1,
    ">30": 0,
    "NO": 0,
}

In [39]:
# keep track of unneeded columns
ID_COLUMNS = [
    "encounter_id",
    "patient_nbr",
]
UNNEEDED_COLUMNS = ID_COLUMNS.copy()

In [40]:
# columns where nan represents no measurements where taken
NOT_MEASURED = 'Not Measured'
NOT_MEASURED_COLUMNS = ['max_glu_serum', 'A1Cresult']

# categorical columns where nan is genuinely unknown
CATEGORICAL_NAN_COLUMNS = ['diag_1', 'diag_2', 'diag_3', 
                           'race', 'medical_specialty', 'payer_code']
UNKNOWN = "Unknown"

# ordinal columns to label as unknown
ORDINAL_NAN_COLUMNS = ['weight']
CATEGORICAL_NAN_COLUMNS += ORDINAL_NAN_COLUMNS

# standardise nan representation
GENDER_COLUMN = "gender"
INVALID_GENDER_LABEL = "Unknown/Invalid"

#### Binary Encode Target Variable

In [41]:
# one hot encoding target variable
UNNEEDED_COLUMNS.append(READMISSION_TARGET_COLUMN)
df_raw[BINARY_TARGET_COLUMN] = (
    df_raw[READMISSION_TARGET_COLUMN]
    .map(TARGET_MAPPING)
)

In [42]:
# check for nans
unexpected_target_values = (
    df_raw[BINARY_TARGET_COLUMN]
    .isna()
    .sum()
)
if unexpected_target_values > 0:
    raise ValueError(
        "Unexpected target labels found during binary conversion."
    )

In [43]:
# summarise target distribution
print("Binary target distribution:")
print(
    df_raw[BINARY_TARGET_COLUMN]
    .value_counts(normalize=True)
)

Binary target distribution:
readmitted_30_days
0    0.888401
1    0.111599
Name: proportion, dtype: float64


#### Check Duplicates

In [44]:
# check dupe rows
total_duplicate_rows = df_raw.duplicated().sum()
print(f"Total duplicate rows: {total_duplicate_rows:,}")

Total duplicate rows: 0


In [45]:
# unique columns
unique_identifier_columns = []
for column in df_raw.columns:
    # compare total rows with unique rows
    unique_value_count = df_raw[column].nunique(dropna=False)
    if unique_value_count == len(df_raw):
        unique_identifier_columns.append(column)

print("Columns with fully unique values per row:")
print(unique_identifier_columns)

Columns with fully unique values per row:
['encounter_id']


#### Keep First Patient Entry

In [46]:
# keep only first encounter for patient nbr
# sort by encounter id first
df_raw = (
    df_raw
    .sort_values("encounter_id")
    .drop_duplicates(
        subset="patient_nbr",
        keep="first",
    )
    .reset_index(drop=True)
)

#### Check NaNs

In [47]:
# calculate nan percentages
missing_value_summary = pd.DataFrame(
    {
        "column": df_raw.columns,
        "missing_count": df_raw.isna().sum().values,
        "missing_percentage": (
            df_raw.isna().sum().values / len(df_raw)
        ) * 100,
    }
)

# sort values
missing_value_summary = (
    missing_value_summary
    .sort_values("missing_percentage", ascending=False)
    .reset_index(drop=True)
)

# filter by non-zero values
missing_value_summary = missing_value_summary[
        missing_value_summary["missing_count"] > 0
]

In [48]:
# summarise missing columns
print("\nColumns with missing values:")
display(missing_value_summary)
MISSING_COLUMNS = list(missing_value_summary['column'].unique())


Columns with missing values:


,column,missing_count,missing_percentage
0,weight,68665,96.010794
1,max_glu_serum,68062,95.167650
2,A1Cresult,58532,81.842333
3,medical_specialty,34477,48.207444
4,payer_code,31043,43.405856
5,race,1948,2.723790
6,diag_3,1225,1.712856
7,diag_2,294,0.411085
8,diag_1,11,0.015381


In [49]:
# summarise dataset nans
print("\nDataset-wide missing values:")
print(f"Total missing values: {df_raw.isna().sum().sum():,}")



Dataset-wide missing values:
Total missing values: 264,257


#### Fill Nans

In [50]:
# configuration for columns
print(MISSING_COLUMNS)
display(df_raw[MISSING_COLUMNS])

['weight', 'max_glu_serum', 'A1Cresult', 'medical_specialty', 'payer_code', 'race', 'diag_3', 'diag_2', 'diag_1']


,weight,max_glu_serum,A1Cresult,medical_specialty,payer_code,race,diag_3,diag_2,diag_1
0,NaN,NaN,NaN,NaN,NaN,Caucasian,38,427,398
1,NaN,NaN,NaN,InternalMedicine,NaN,Caucasian,486,198,434
2,NaN,NaN,NaN,NaN,NaN,Caucasian,250,157,197
3,NaN,NaN,NaN,NaN,NaN,AfricanAmerican,996,403,250.7
4,NaN,NaN,NaN,NaN,NaN,Caucasian,250,411,414
...,...,...,...,...,...,...,...,...,...
71513,NaN,NaN,>7,NaN,NaN,Caucasian,250.02,574,574
71514,NaN,NaN,>8,NaN,MD,Other,518,599,592
71515,NaN,NaN,NaN,NaN,MD,Other,403,585,996
71516,NaN,NaN,NaN,NaN,MC,Caucasian,304,8,292


In [51]:
# fill nans that meaningfully mean no measurements were taken
for col in NOT_MEASURED_COLUMNS:
    df_raw[col] = df_raw[col].fillna(NOT_MEASURED)
    MISSING_COLUMNS.remove(col)

In [52]:
# fill remaining categorical columns with unknown marker
for col in CATEGORICAL_NAN_COLUMNS:
    df_raw[col] = df_raw[col].fillna(UNKNOWN)
    MISSING_COLUMNS.remove(col)

In [53]:
# standardise invalid gender labels into a cleaner category.
df_raw[GENDER_COLUMN] = df_raw[GENDER_COLUMN].replace(
    INVALID_GENDER_LABEL, UNKNOWN
)

#### Drop Redundant Columns

In [54]:
df_clean = df_raw.drop(columns=UNNEEDED_COLUMNS)

#### Save Clean Data

In [55]:
# save data
def save_data(df, OUTPUT_FILENAME, OUTPUT_FOLDER):
    NOTEBOOK_DIRECTORY = Path.cwd()
    DATA_DIRECTORY = (
        NOTEBOOK_DIRECTORY / f"../Data/{OUTPUT_FOLDER}"
    ).resolve()

    OUTPUT_PATH = (
        DATA_DIRECTORY / OUTPUT_FILENAME
    )
    df.to_csv(
        OUTPUT_PATH, index=False 
    )

In [56]:
# saved config
OUTPUT_FILENAME = "diabetes_clean.csv"
OUTPUT_FOLDER = "Cleaned"

# save data
save_data(df_clean, OUTPUT_FILENAME, OUTPUT_FOLDER)

## Stratify Dataset

#### Stratification Configuration

In [57]:
# stratification parameters
NUMBER_OF_FOLDS = 10
RANDOM_SEED = 42
FOLD_COLUMN = 'fold'


#### Apply Stratification

In [58]:
def create_stratified_k_folds(
    dataset: pd.DataFrame,
    target_column: str,
    number_of_folds: int,
    random_seed: int = RANDOM_SEED) -> list[pd.DataFrame]:
    """
    Create stratified k-fold splits with approximately equal class balance
    """
    rng = np.random.default_rng(random_seed)
    stratified_folds = [
        [] for _ in range(number_of_folds)
    ]

    # Split each class separately so proportions remain balanced.
    for target_value in sorted(dataset[target_column].unique()):
        # filter by target value
        class_subset = dataset[
            dataset[target_column] == target_value
        ].sample(frac=1, random_state=random_seed)

        # randomly segment        
        shuffled_indices = class_subset.index.to_numpy()
        rng.shuffle(shuffled_indices)

        # split by folds and extend existing dataframes
        split_indices = np.array_split(
            shuffled_indices, number_of_folds
        )
        for fold_index in range(number_of_folds):
            stratified_folds[fold_index].extend(
                split_indices[fold_index]
            )

    # aggregate fold results    
    final_folds = []
    for fold_indices in stratified_folds:
        fold_dataframe = (
            dataset
            .loc[fold_indices]
            .sample(frac=1, random_state=random_seed)
            .reset_index(drop=True)
        )
        final_folds.append(fold_dataframe)
    return final_folds

In [59]:
# apply function
folds_df_clean = create_stratified_k_folds(
    dataset=df_clean,
    target_column=BINARY_TARGET_COLUMN,
    number_of_folds=NUMBER_OF_FOLDS,
    random_seed=RANDOM_SEED,
)

In [60]:
# label folds
for i in range(len(folds_df_clean)):
    folds_df_clean[i][FOLD_COLUMN] = i
df_fold = pd.concat(folds_df_clean)

In [61]:
# Move fold and target columns to the front
remaining_columns = [
    column
    for column in df_fold.columns
    if column not in [FOLD_COLUMN, BINARY_TARGET_COLUMN]
]

df_fold = df_fold[
    [FOLD_COLUMN, BINARY_TARGET_COLUMN] + remaining_columns
].copy()

#### Save Folds

In [62]:
# save folds
OUTPUT_FOLDER = 'Stratified'
OUTPUT_FILENAME = f'diabetes_stratified.csv'
save_data(df_fold, OUTPUT_FILENAME, OUTPUT_FOLDER)